In [18]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"

# Your custom text color
mycolor = 'black'  # or e.g., '#fe9463', (0.996, 0.580, 0.388)
from matplotlib import rcParams
# Global style settings
rcParams.update({
    # Enable LaTeX rendering
    'text.usetex': True,

    # Font settings
    'font.family': 'sans-serif',
    'font.sans-serif': ['Open Sans'],  # Use Open Sans if available

    # Set all text and visual elements to use the custom color
    'text.color': mycolor,
    'axes.labelcolor': mycolor,
    'xtick.color': mycolor,
    'ytick.color': mycolor,
    'axes.edgecolor': mycolor,
    'grid.color': mycolor,
    'legend.edgecolor': mycolor,
    'legend.facecolor': 'none',
    'legend.labelcolor': mycolor,

    # Transparent backgrounds
    'figure.facecolor': 'none',
    'axes.facecolor': 'none',
})

if mycolor=='white':
    rcParams['savefig.transparent'] = True

import matplotlib as mpl
mpl.rcParams['figure.facecolor'] = 'none'
mpl.rcParams['axes.facecolor']   = 'none'
mpl.rcParams['savefig.facecolor'] = 'none'
# fix tick labels size
mpl.rcParams['xtick.labelsize'] = 21
mpl.rcParams['ytick.labelsize'] = 21


## Horizon data
Calculate the horizon with GWFish

In [13]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import GWFish.modules as gw

In [14]:
# placeholder parameters for sources
parameters_src = {
        'mass_1_source': 30,  
        'mass_2_source': 30,
        'theta_jn': 0.,
        'ra': 3.5,
        'dec': -0.5,
        'psi': 0.,
        'phase': 0.,
        'geocent_time': 0.
    }

In [15]:
###########################################################
# REMEMBER to put absolute paths to psds files in the
# yaml configuration file!
###########################################################
myconfig = os.path.join('../../gw_data/my_detectors.yaml')
#Einstein Telescope Sensitivity (ET_10_full_cryo_psd.txt, public link: https://apps.et-gw.eu/tds/?r=18213)
network = gw.detection.Network(['ET_cryo_10'], config=myconfig)

In [19]:
# vary masses to pass as input to the horizon function
tot_mass = np.logspace(0., 4, 100)
redshifts_snr8_et = []
redshifts_snr150_et = []

for mass in tqdm(tot_mass):
    # use equal mass systems
    parameters_src['mass_1_source'] = mass/2
    parameters_src['mass_2_source'] = mass/2
    output = gw.horizon.find_optimal_location(parameters_src, network, waveform_model='IMRPhenomD')
    distance_snr8_et, redshift_snr8_et = gw.horizon.horizon(output, network, target_SNR=8, waveform_model='IMRPhenomD')
    redshifts_snr8_et.append(redshift_snr8_et)
    distance_snr150_et, redshift_snr150_et = gw.horizon.horizon(output, network, target_SNR=150, waveform_model='IMRPhenomD')
    redshifts_snr150_et.append(redshift_snr150_et)

100%|██████████| 100/100 [04:58<00:00,  2.99s/it]


In [20]:
# save horizon data to a file
horizon_data = pd.DataFrame()
horizon_data['Total_Source_Frame_Mass_SolarMass'] = tot_mass
horizon_data['Redshift_Horizon_ET_snr8'] = redshifts_snr8_et
horizon_data['Redshift_Horizon_ET_snr150'] = redshifts_snr150_et
horizon_data.to_csv('horizon_data_et.csv', index=False)